### QQQ - metadata extraction tests

In [0]:
from pyspark.sql import SparkSession
from src.etl.extraction.qqq_categories_extraction import QQQCategoriesExtractor

spark = SparkSession.builder.getOrCreate()

symbols = ["AAPL", "MSFT", "NVDA"]

extractor = QQQCategoriesExtractor(spark)
df = extractor.extract(symbols)

display(df)

### Fred silver table EDA

In [0]:
%sql
SELECT COUNT (*) AS total_rows
FROM silver.fred_indicators

In [0]:
%sql
SELECT 
    COUNT(*) FILTER (WHERE unit IS NULL) AS unit_total_nulls,
    COUNT(*) FILTER (WHERE frequency IS NULL) AS frequenct_total_nulls
FROM silver.fred_indicators;

In [0]:
from pyspark.sql.functions import col, count
silver_macro_indicators = 'workspace.silver.fred_indicators'

df = spark.table(silver_macro_indicators)

duplicates = (
    df.groupBy("indicator_id", "date")
      .agg(count("*").alias("cnt"))
      .filter(col("cnt") > 1)
)

duplicates.show()

In [0]:
%sql
SELECT DISTINCT indicator_id
from silver.fred_indicators

In [0]:
%sql
select distinct indicator_id
from silver.fred_macro_metadata_indicators

In [0]:
%sql
TRUNCATE TABLE silver.fred_macro_metadata_indicators;

### extraction yfinance API tests

In [0]:
%pip install yfinance

In [0]:
import yfinance as yf


def validate_symbol_exists(symbol: str, lookback_days: int = 5) -> bool:
    """
    Sprawdza czy symbol istnieje i zwraca dane historyczne.
    Zwraca True jeśli dane dostępne, False jeśli brak.
    """

    try:
        ticker = yf.Ticker(symbol)
        df = ticker.history(period=f"{lookback_days}d")

        if df.empty:
            print(f"[WARN] {symbol} - brak danych")
            return False

        print(f"[OK] {symbol} - dane dostępne ({len(df)} rekordów)")
        return True

    except Exception as e:
        print(f"[ERROR] {symbol} - wyjątek: {e}")
        return False


# ===== TEST =====

symbols = ["aapl"]

for s in symbols:
    validate_symbol_exists(s)

In [0]:
import yfinance as yf

def get_symbol_start_date(symbol: str):
    try:
        ticker = yf.Ticker(symbol)
        df = ticker.history(period="max")

        if df.empty:
            print(f"[WARN] {symbol} - brak danych")
            return None

        first_date = df.index.min().date()
        print(f"[OK] {symbol} - dane od: {first_date}")
        return first_date

    except Exception as e:
        print(f"[ERROR] {symbol} - wyjątek: {e}")
        return None

symbols = [
    "NVDA","AAPL","MSFT","AVGO","AMZN","GOOGL","TSLA","GOOG","META","NFLX",
    "PLTR","AMD","COST","CSCO","MU","TMUS","SHOP","LRCX","PEP","LIN",
    "QCOM","APP","ISRG","AMAT","INTU","INTC","BKNG","KLAC","AMGN","GILD",
    "PANW","TXN","ADBE","CRWD","HON","MELI","CEG","ADI","VRTX","ADP",
    "DASH","CMCSA","SBUX","CDNS","ASML","PDD","SNPS","ORLY","MRVL","MDLZ",
    "CTAS","MAR","TRI","REGN","MSTR","CSX","AEP","MNST","PYPL","FTNT",
    "ADSK","AXON","ABNB","WBD","NXPI","PCAR","ROST","DDOG","WDAY","IDXX",
    "ZS","EA","AZN","XEL","FAST","BKR","ROP","EXC","TTWO","PAYX",
    "FANG","CPRT","CCEP","KDP","USD","CTSH","GEHC","MCHP","CHTR","VRSK",
    "CSGP","KHC","ODFL","DXCM","TEAM","ARM","BIIB","TTD","ON","CDW"
]

for s in symbols:
    get_symbol_start_date(s)


In [0]:
import yfinance as yf
import pandas as pd

symbols = ["AAPL"]
start_date = "1990-01-01"
end_date = "2026-02-01"

all_data = []

for symbol in symbols:
    df = yf.download(
        symbol,
        start=start_date,
        end=end_date,
        interval="1d",
        auto_adjust=False
    )
    
    df = df.reset_index()
    df["symbol"] = symbol

    all_data.append(df)

final_df = pd.concat(all_data)

spark_df = spark.createDataFrame(final_df)




### Cleaning yfinance 

In [0]:
from src.config.config_loader import load_config
from pyspark.sql import SparkSession
from src.etl.cleaning.ohlcv_cleaning import OHLCVCleaner

spark = SparkSession.builder.getOrCreate()

config = load_config()

qqq_silver_entities = config["tables"]["silver_qqq"]

table = spark.table(qqq_silver_entities)

symbols = [
    row["symbol"]
    for row in table.select("symbol").distinct().collect()
]

cleaned_symbols = OHLCVCleaner.clean_list(symbols)

len(cleaned_symbols)


### yfinance OHLCV(Exploratory Data Analysis) - EDA

In [0]:
df = spark.table("bronze.ohlcv_indicators")
df.printSchema()

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import col


df = df.withColumn("date", col("date").cast("date")) \
    .withColumn("open", col("open").cast("double")) \
    .withColumn("high", col("high").cast("double")) \
    .withColumn("low", col("low").cast("double")) \
    .withColumn("close", col("close").cast("double")) \
    .withColumn("adj_close", col("adj_close").cast("double")) \
    .withColumn("volume", col("volume").cast("long"))

display(df.limit(10))



In [0]:
df.count()

In [0]:
from pyspark.sql import functions as F

df.groupBy("symbol").count().show(105)

In [0]:
from pyspark.sql import functions as F

df.groupBy("symbol") \
  .agg(
      F.min("date").alias("min_date"),
      F.max("date").alias("max_date")
  ) \
    .orderBy(F.col("min_date").desc()) \
  .show(300)

In [0]:
from pyspark.sql.functions import count

df.groupBy("symbol", "date") \
  .agg(count("*").alias("cnt")) \
  .filter("cnt > 1") \
  .show()

In [0]:
from pyspark.sql.functions import col, sum

df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
]).show()

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import lag, datediff

df = spark.table("bronze.ohlcv_indicators")

w = Window.partitionBy("symbol").orderBy("date") #tworzy grupy na podstawie tickerów kolumny symbol i sortuje po kolumnie date

df_with_gap = (
    df.withColumn("prev_date", lag("date").over(w))
      .withColumn("gap_days", datediff("date", "prev_date"))
) # dodaje date do kolumny prev_date na podstawie realnej daty tickera i w kolunmg gap_days oblicza różnicę między datami

df_with_gap.filter("gap_days > 4").count() # filtruje rekordy gdzie gap_days jest większy niż 3

In [0]:
from pyspark.sql.functions import min, max, col

agg_df = (
    df
    .groupBy("symbol")
    .agg(
        min("date").alias("min_date"),
        max("date").alias("max_date")
    )
)

agg_df.show(20)

# jeśli max_date == min_date → tylko 1 dzień danych
if agg_df.filter(col("max_date") == col("min_date")).count() > 0:
    raise ValueError("Symbol with only one date found")

In [0]:
from pyspark.sql.functions import col

df.groupBy("symbol").count().filter(col("count") < 2).count()

other

In [0]:


cleaned_df = spark.read.format("delta").load("/Volumes/workspace/bronze/temp_table")

metadata_df = spark.table("workspace.silver.fred_macro_metadata_indicators")

cleaned_df.show()



In [0]:
%sql
SHOW SCHEMAS IN workspace;

In [0]:
%sql
DROP TABLE IF EXISTS workspace.bronze.qqq_etf_entities_enriched;

### New symbols in ohlcv pipeline

In [0]:
df = spark.table("bronze.ohlcv_indicators")


In [0]:
print(df.select("symbol").distinct().count())

In [0]:
df.filter(df.symbol.isin("^VIX", "QQQ", "SPY", "TLT", "GLD")).select("symbol").distinct().show()


In [0]:
# ile rekordów per benchmark
df.filter(df.symbol.isin("^VIX", "QQQ", "SPY", "TLT", "GLD")).groupBy("symbol").count().show()

In [0]:
non_benchmark = df.filter(~col("symbol").isin("^vix", "qqq", "spy", "tlt", "gld"))
non_benchmark.filter(col("name").isNull()).select("symbol").distinct().show()

In [0]:
import requests
r = requests.get("https://api.gdeltproject.org/api/v2/doc/doc?query=%22NASDAQ%22&mode=artlist&maxrecords=5&format=json&timespan=1d")
print(r.status_code)

### AV_sentiment

In [0]:
dbutils.secrets.get("my-scope", "API_KEY_AV")

In [0]:
from pyspark.sql.functions import col

entities_df = spark.table("gold.ohlcv_with_dimension")

symbols = [
    row["symbol"].upper()
    for row in (
        entities_df
        .filter(col("sector") == "technology")
        .orderBy(col("percent_holding").desc())
        .select("symbol")
        .distinct()
        .limit(50)
        .collect()
    )
]

tickers = ",".join(symbols)

print(tickers)

In [0]:
import requests, json

response = requests.get("https://www.alphavantage.co/query?function=NEWS_SENTIMENT&tickers=AAPL&apikey=demo")
av_json = response.json()
print(json.dumps(av_json["feed"][0], indent=2))

In [0]:
article = av_json["feed"][0]
ticker_data = article["ticker_sentiment"][0]

row = {
    "ticker": ticker_data["ticker"],
    "time_published": article["time_published"],
    "source": article["source"],
    "title": article["title"],
    "relevance_score": ticker_data["relevance_score"],
    "ticker_sentiment_score": ticker_data["ticker_sentiment_score"],
    "ticker_sentiment_label": ticker_data["ticker_sentiment_label"],
    "overall_sentiment_score": article["overall_sentiment_score"],
    "overall_sentiment_label": article["overall_sentiment_label"],
}

print(json.dumps(row, indent=2))

In [0]:
spark.table("bronze.av_sentiment").write.format("delta").mode("overwrite").saveAsTable("bronze.av_sentiment_backup")